### Image Stitching
- Detect keypoints in all of images
- Match the descriptors between two images
- Use RANSAC Algo to estimate a homography matrix using matched descriptors
- Apply wrap transformation using the estimated HM

## 1. Import Statements

In [1]:
import cv2
import numpy as np
import os

In [2]:
image_paths = os.listdir('Images/Stitching')

In [3]:
image_paths

['1.jpg', '2.jpg', '3.jpg', '4.jpg']

In [4]:
images = []

for image in image_paths:
    img = cv2.imread('Images/Stitching/'+image)
    img = cv2.resize(img, (600,600))
    images.append(img)
    cv2.imshow('Image', img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [5]:
cv2.destroyAllWindows()

In [6]:
images[0].shape

(600, 600, 3)

## 2. Image Stitcher

In [7]:
imageStitcher = cv2.Stitcher_create()

error, stitched_img = imageStitcher.stitch(images)

if not error:
    cv2.imwrite('Images/Stitching/Stitch_Output.png', stitched_img)
    cv2.imshow('Stitched Image', stitched_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    stitched_img = cv2.copyMakeBorder(stitched_img, 10,10,10,10, cv2.BORDER_CONSTANT, (0,0,0))
    
    gray = cv2.cvtColor(stitched_img, cv2.COLOR_BGR2GRAY)
    thresh_img = cv2.threshold(gray, 0 ,255, cv2.THRESH_BINARY)[1]
    
    cv2.imshow('Threshold Image', thresh_img)
    cv2.waitKey(0)
    
    contours, hier = cv2.findContours(thresh_img.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    areaOI = max(contours, key=cv2.contourArea) 
    
    mask = np.zeros(thresh_img.shape, dtype='uint8')
    x, y, w, h = cv2.boundingRect(areaOI)
    cv2.rectangle(mask, (x,y), (x+w, y+h), 255, -1)
    
    minRectangle = mask.copy()
    sub = mask.copy()
    
    while cv2.countNonZero(sub) > 0:
        minRectangle = cv2.erode(minRectangle, None)
        sub = cv2.subtract(minRectangle, thresh_img)
        
    contours, hier = cv2.findContours(minRectangle.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    areaOI = max(contours, key=cv2.contourArea)
    
    cv2.imshow('minRectImage', minRectangle)
    cv2.waitKey(0)
    
    x, y, w, h = cv2.boundingRect(areaOI)
    
    stitched_img = stitched_img[y:y+h, x:x+w]
    
    cv2.imwrite('Images/Stitching/Stitch_Processed.png', stitched_img)
    cv2.imshow('Stitched Image', stitched_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    
    

In [8]:
cv2.destroyAllWindows()